### Mouse movement demo

In [1]:
import logging
import time
from PyQt5 import QtCore, QtWidgets

# Configure logging to output the message in the required format.
logging.basicConfig(level=logging.INFO, format='%(message)s')

class MouseDragLoggerWidget(QtWidgets.QWidget):
    def __init__(self, parent=None):
        super(MouseDragLoggerWidget, self).__init__(parent)
        self._drag_active = False
        # Enable mouse tracking even when no button is pressed (optional)
        self.setMouseTracking(True)

    def mousePressEvent(self, event):
        if event.button() == QtCore.Qt.LeftButton:
            self._drag_active = True
            current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
            pos = event.pos()
            logging.info(f"start_drag: {current_time} <{pos.x()}, {pos.y()}>")
        # Always call the base class method if you need default behavior.
        super(MouseDragLoggerWidget, self).mousePressEvent(event)

    def mouseMoveEvent(self, event):
        if self._drag_active:
            current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
            pos = event.pos()
            logging.info(f"drag: {current_time} <{pos.x()}, {pos.y()}>")
        super(MouseDragLoggerWidget, self).mouseMoveEvent(event)

    def mouseReleaseEvent(self, event):
        if self._drag_active and event.button() == QtCore.Qt.LeftButton:
            current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
            pos = event.pos()
            logging.info(f"end_drag: {current_time} <{pos.x()}, {pos.y()}>")
            self._drag_active = False
        super(MouseDragLoggerWidget, self).mouseReleaseEvent(event)

In [2]:
import sys

app = QtWidgets.QApplication(sys.argv)
window = MouseDragLoggerWidget()
window.resize(400, 300)
window.setWindowTitle("Mouse Drag Logger")
window.show()
sys.exit(app.exec_())

2025-02-26 16:11:50.198 Python[76564:2189419] +[IMKClient subclass]: chose IMKClient_Modern
2025-02-26 16:11:50.198 Python[76564:2189419] +[IMKInputSession subclass]: chose IMKInputSession_Modern
start_drag: 2025-02-26 16:11:52 <59, 57>
drag: 2025-02-26 16:11:52 <59, 57>
drag: 2025-02-26 16:11:52 <59, 58>
drag: 2025-02-26 16:11:52 <60, 58>
drag: 2025-02-26 16:11:52 <60, 58>
drag: 2025-02-26 16:11:52 <60, 58>
drag: 2025-02-26 16:11:52 <60, 58>
drag: 2025-02-26 16:11:53 <60, 58>
drag: 2025-02-26 16:11:53 <60, 58>
drag: 2025-02-26 16:11:53 <60, 58>
drag: 2025-02-26 16:11:53 <61, 58>
drag: 2025-02-26 16:11:53 <61, 59>
drag: 2025-02-26 16:11:53 <61, 59>
drag: 2025-02-26 16:11:53 <61, 59>
drag: 2025-02-26 16:11:53 <61, 59>
drag: 2025-02-26 16:11:53 <61, 59>
drag: 2025-02-26 16:11:53 <62, 59>
drag: 2025-02-26 16:11:53 <62, 60>
drag: 2025-02-26 16:11:53 <62, 60>
drag: 2025-02-26 16:11:53 <62, 60>
drag: 2025-02-26 16:11:53 <62, 60>
drag: 2025-02-26 16:11:53 <62, 60>
drag: 2025-02-26 16:11:53 <6

SystemExit: 0

/Users/guangyaoquan/Downloads/3DSlicer/registrationViewer/slicer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Mouse log to dataframe

In [ ]:
import pandas as pd
import re

def process_mouse_logs(file_path):
    # Patterns for different log entries
    detailed_pattern = re.compile(
        r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}) ~ \w+ ~ INFO ~ \w+ ~ \w+ ~ \w+ ~ MOUSE ~ (?P<action>[\w\s]+): (?P<view>[\w\d]+)(?: - (?P<delta>delta=[-\d]+)| \((?P<x>\d+), (?P<y>\d+)\))?"
    )
    simple_pattern = re.compile(
        r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}).*MOUSE ~ (?P<action>Zoom|Scroll|Window_Level|Pan): \((?P<coords>\d+, \d+)\)"
    )
    end_pattern = re.compile(
        r"(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}).*MOUSE ~ End (?P<action>[\w\s]+)"
    )

    detailed_data, simple_data, end_data = [], [], []

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            detailed_match = detailed_pattern.match(line)
            simple_match = simple_pattern.search(line)
            end_match = end_pattern.search(line)

            if detailed_match:
                data = detailed_match.groupdict()
                value = data["delta"] if data["delta"] else (f"({data['x']}, {data['y']})" if data["x"] else "NaN")
                detailed_data.append({"timestamp": data["timestamp"], "action": data["action"].strip(), "view": data["view"], "value": value})

            if simple_match:
                data = simple_match.groupdict()
                simple_data.append({"timestamp": data["timestamp"], "action": data["action"], "value": f"({data['coords']})", "view": pd.NA})

            if end_match:
                data = end_match.groupdict()
                end_data.append({"timestamp": data["timestamp"], "action": f"End {data['action'].strip()}", "view": pd.NA, "value": pd.NA})

    # Convert to DataFrames
    df_detailed = pd.DataFrame(detailed_data)
    df_simple = pd.DataFrame(simple_data)
    df_end = pd.DataFrame(end_data)

    # Combine DataFrames
    merged_df = pd.concat([df_detailed, df_simple, df_end], ignore_index=True)
    merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], format="%Y-%m-%d %H:%M:%S,%f")
    merged_df.sort_values(by='timestamp', inplace=True)

    # Forward-fill 'view' based on Start events
    merged_df['view'] = merged_df['view'].ffill()

    # Save to CSV
    merged_df.to_csv("merged_mouse_events.csv", index=False)

    return merged_df

file_path = "/Users/guangyaoquan/Downloads/rad_1.log"
df_merged = process_mouse_logs(file_path)
print("Merged mouse events saved to merged_mouse_events.csv")

Merged mouse events saved to merged_mouse_events.csv
